In [19]:
!pip install faiss-cpu sentence-transformers requests beautifulsoup4

In [22]:
import requests
from bs4 import BeautifulSoup
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [23]:
url = "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml"

response = requests.get(url)
soup = BeautifulSoup(response.content, "xml")

docs = []

for item in soup.find_all("item")[:15]:
    title = item.title.text if item.title else ""
    link = item.link.text if item.link else ""
    description = item.description.text if item.description else ""

    text = f"Title: {title}\nDescription: {description}"

    if len(text.strip()) > 50:
        docs.append({
            "title": title,
            "link": link,
            "text": text
        })

print("Total usable docs:", len(docs))

Total usable docs: 15


In [24]:
print(docs[0]["title"])
print()
print(docs[0]["text"])

Judge Stays Pentagon’s Labeling of Anthropic as ‘Supply Chain Risk’

Title: Judge Stays Pentagon’s Labeling of Anthropic as ‘Supply Chain Risk’
Description: The decision is an early victory for the artificial intelligence company in a rancorous legal battle with the Department of Defense.


In [25]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["text"] for doc in docs]
embeddings = model.encode(texts)

print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of embeddings: 15
Embedding size: 384


In [26]:
embedding_array = np.array(embeddings).astype("float32")

dimension = embedding_array.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embedding_array)

print("FAISS index size:", index.ntotal)

FAISS index size: 15


In [27]:
query = "latest AI news"

query_embedding = model.encode([query]).astype("float32")

D, I = index.search(query_embedding, k=3)

print("Indexes:", I)

Indexes: [[9 7 3]]


In [28]:
for idx in I[0]:
    print("TITLE:", docs[idx]["title"])
    print("TEXT:", docs[idx]["text"])
    print("LINK:", docs[idx]["link"])
    print("-" * 80)

TITLE: In ‘The AI Doc,’ Sam Altman and Dario Amodei Go on the Record
TEXT: Title: In ‘The AI Doc,’ Sam Altman and Dario Amodei Go on the Record
Description: “The AI Doc: Or How I Became an Apocaloptimist” tries to cover so much that it ends up being more confusing than clarifying, but parts are fascinating.
LINK: https://www.nytimes.com/2026/03/26/movies/the-ai-doc-movie-anthropic-openai-claude-chatgpt.html
--------------------------------------------------------------------------------
TITLE: Shield AI, a Start-Up Making Military Drones, Raises $2 Billion
TEXT: Title: Shield AI, a Start-Up Making Military Drones, Raises $2 Billion
Description: The company, which develops autonomous military technology, also plans to buy a maker of simulation software as interest in next-generation defense soars.
LINK: https://www.nytimes.com/2026/03/26/business/dealbook/shield-ai-drones-aechelon-fund-raising.html
--------------------------------------------------------------------------------
TITLE: T